# GeoSR-4 — EDSR baseline training (Colab GPU)

Runs the same code that was smoke-tested locally on CPU (see `ml/training/train_edsr.py`, `decisions.md` D009 for a gradient-flow bug that was fixed before this run). This notebook just provides the GPU compute — no model/pipeline logic lives here.

**Before running:** Runtime → Change runtime type → GPU.

In [ ]:
!nvidia-smi

## 1. Clone the repo and install dependencies
Colab already ships CUDA-enabled PyTorch, so we only add the packages it's missing.

In [ ]:
!git clone https://github.com/Vijay6923/GeoSR-4.git
%cd GeoSR-4
!pip install -q rasterio huggingface_hub scikit-image

## 2. Download the dataset
Only the cross-sensor split (~2.1 GB, real Sentinel-2<->NAIP pairs) -- see `decisions.md` D002/D003 for why.

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, os

zip_path = hf_hub_download(
    repo_id="isp-uv-es/SEN2NAIP",
    repo_type="dataset",
    filename="cross-sensor/cross-sensor.zip",
    local_dir="ml/datasets/raw/sen2naip",
)

extract_dir = "ml/datasets/raw/sen2naip/cross-sensor/extracted"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract_dir)

print("extracted to", extract_dir)

## 3. Confirm normalization stats came from the repo (not recomputed)
`configs/normalization_stats.json` is committed -- it must be reused as-is so train/val/test all normalize the same way. Don't rerun `compute_stats.py` here.

In [ ]:
import json
print(json.load(open("configs/normalization_stats.json")))

## 4. Train
Full EDSR-baseline config (16 blocks, 64 channels). Adjust `--epochs`/`--batch-size` for your GPU/time budget.

In [ ]:
!python ml/training/train_edsr.py \
  --epochs 20 \
  --batch-size 16 \
  --n-blocks 16 \
  --n-channels 64 \
  --lr 1e-4 \
  --checkpoint-dir experiments/edsr \
  --log-every 20

## 5. Save checkpoints somewhere durable
Colab's local disk disappears when the runtime recycles -- copy the checkpoint out before you close this.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/GeoSR-4-checkpoints
!cp experiments/edsr/*.pt /content/drive/MyDrive/GeoSR-4-checkpoints/